# Fig. 9 — deviation from hexagonal packing, heatmaps (panels A–D)

Run from the project root (the save path is relative).

In [ ]:
from vertex_lite.custom import load_data
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import numpy as np
import pyprojroot.here as here

## Load Sort and Filter

In [ ]:
def filter_data(time,local=False,cluster=False):
    if cluster:
        rpath=here('results/resultados_cluster')
    else:
        rpath=here('results/resultados_random')

    df=load_data(f'{rpath}/noise_*/pct_*/sim_*/cells.parquet',columns=['time','n_lados','neighbor_of_mutant','cell_id','alive'],time_filter=time)
    if df.empty:
        return print(f"No data found for time {time} and local {local}")
    df = df[df['alive'] == 1]
    if local:
        df=df[df['neighbor_of_mutant'] == True]
    else:
        df=df.drop(columns=['neighbor_of_mutant'])

    df = (
        df[df['n_lados'] >= 3]
        .groupby(['time','mesh_type','pct_mutant','n_lados'])
        .size()
        .reset_index(name='n_cells')
    )
    df['total_cells'] = (
        df.groupby(['time','mesh_type','pct_mutant'])['n_cells']
        .transform('sum')
    )
    df['frac_cells'] = df['n_cells'] / df['total_cells']
    return df

## Plotting

In [ ]:
def get_heatmap_data(time, local=False, case="hex", cluster=False):
    """Mean absolute deviation from the hexagon, <|n - 6|>, per (D, f_mut)."""
    df = filter_data(time, local, cluster)
    grouped = df.groupby(["mesh_type", "pct_mutant"])[['n_lados', 'n_cells', 'frac_cells']]

    title = "Mean Absolute Deviation from Hexagon"

    def mad_hex(group):
        N = group["n_cells"].sum()
        return (group["n_cells"] * abs(group["n_lados"] - 6)).sum() / N

    heatmap_data = grouped.apply(mad_hex).unstack()

    heatmap_data = heatmap_data.sort_index().sort_index(axis=1)
    heatmap_data = heatmap_data.T

    if local and 0.0 not in heatmap_data.index:
        heatmap_data.loc[0.0] = 0

    heatmap_data.index = pd.to_numeric(heatmap_data.index)
    heatmap_data = heatmap_data.sort_index()

    return heatmap_data, title


def plot_heatmap(heatmap_data, title, time, local=False, case=None, cluster=False):
    cmap = sns.light_palette("#EA5A00", as_cmap=True)
    sns.set_theme(style="white", context="talk")

    plt.rcParams.update({
        "font.size": 20,
        "axes.labelsize": 25,
        "axes.titlesize": 25,
        "legend.fontsize": 12,
        "axes.linewidth": 0.0,
        "xtick.labelsize": 24,
        "ytick.labelsize": 24,
    })
    fig, ax = plt.subplots(figsize=(8, 6))

    im = ax.imshow(
        heatmap_data.values,
        cmap=cmap,
        origin='lower',  # first row at the bottom
        aspect='auto',
        vmin=0,
        vmax=1.0
    )

    # --- AXES LABELS ---
    step = 2
    plt.tick_params(axis='both', labelsize=15)
    # X-axis: mesh distortion (columns)
    x_ticks = np.arange(0, len(heatmap_data.columns), step)
    ax.set_xticks(x_ticks)
    ax.set_xticklabels(heatmap_data.columns[::step], rotation=45)

    # Y-axis: % mutants (index)
    y_ticks = np.arange(0, len(heatmap_data.index), step)
    ax.set_yticks(y_ticks)
    ax.set_yticklabels(heatmap_data.index[::step])

    # --- COLORBAR ---
    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label(case)

    ax.annotate(
        f'T={time} | local={local}',
        xy=(0.0, -0.2),
        xycoords='axes fraction',
        ha='left',
        fontsize=10
    )

    plt.xlabel("Initial Mesh Distortion")
    plt.ylabel("% mutants")
    plt.title(f'{title} for Neighbors of Mutant Cells' if local else f'{title} for Whole Tissue')

    plt.tight_layout()

    suffix = "local" if local else "global"
    clustering = "cluster" if cluster else "random"

    savedir = "results/polygons/plots/pol_heatmaps"
    os.makedirs(savedir, exist_ok=True)

    filename = f"{savedir}/heatmap_{clustering}_time_{time}_{case}_{suffix}.svg"

    fig.savefig(filename, format='svg')
    plt.close(fig)

## Figures

In [ ]:
# Heatmaps evaluated at t = 15000 (last saved time step, 14980)
times = [[14980], [14980]]

data_cache = {}

for c in ["hex"]:
    for l in [True, False]:
        # Random
        for t in times[0]:
            key = (t, l, c, False)  # (time, local, case, cluster)
            print(f"Analyzing {key}...")
            data_cache[key] = get_heatmap_data(time=t, local=l, case=c, cluster=False)

        # Cluster
        for t in times[1]:
            key = (t, l, c, True)
            print(f"Analyzing {key}...")
            data_cache[key] = get_heatmap_data(time=t, local=l, case=c, cluster=True)

print("All data loaded into memory.")

In [ ]:
for key, (heatmap_df, title) in data_cache.items():
    t, l, c, is_cluster = key
    plot_heatmap(heatmap_df, title, t, local=l, case=c, cluster=is_cluster)